In [12]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgbm
import xgboost as xgb
import torch

print(torch.cuda.is_available())

from functools import partial

import sys
sys.path.append('../src')

from load_clean_data import load_clean_data
from backtest import get_cutoffs, split_data_by_cutoff, run_backtest
from baselines import add_all_baselines
from evaluation import score_forecasts, merge_backtests
from features import build_features, FEATURE_COLUMNS
from models import add_lgbm_forecast, LGBM_PARAMS, add_xgb_forecast, XGB_PARAMS
from tuning import score_params, sample_param, random_search
from nn_models import prepare_inputs, add_nn_forecast, NN_PARAMS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
True


In [2]:
df = load_clean_data()
df_with_features = build_features(df)

In [ ]:
cutoffs = get_cutoffs(df['Date'].max(), 7)

In [5]:
FORECAST = True

if FORECAST:

    bt = run_backtest(df_with_features, cutoffs, 7, add_nn_forecast)

    nn_bt = bt[['Store', 'Date', 'Sales', 'cutoff', 'day_ahead', 'nn_forecast']]

    nn_bt.to_csv('../results/backtests/nn_backtest.csv', index=False)

else: 
    nn_bt = pd.read_csv('../results/backtests/nn_backtest.csv', parse_dates=['Date', 'cutoff'], low_memory=False)

In [6]:
SAVE_SCORES = True

if SAVE_SCORES:
    scores = score_forecasts(nn_bt, ['nn_forecast'])
    scores.to_csv('../results/nn_scores.csv', index=False)

In [7]:
first_10 = cutoffs[:10]

In [8]:
NN_SEARCH_SPACE = {'epochs': ('int', 5, 40),
                   'learning_rate':	('log',	0.0003, 0.01),
                   'hidden_size':	('int',	64, 512),
                   'n_layers':	('int',	1, 4),
                   'dropout':	('float', 0.0, 0.3),
                   'weight_decay':	('log',	0.000001, 0.01),
                   'store_embed_dim':	('int',	4, 32),
}

In [11]:
RUN_SEARCH = True

if RUN_SEARCH:
    nn_tuning = random_search(df_with_features, first_10, add_nn_forecast, 'nn_forecast', NN_PARAMS, NN_SEARCH_SPACE, n_trials=25, path='../results/tuning/nn_tuning.csv')

    print(nn_tuning.head(5))

else:
    nn_tuning = pd.read_csv('../results/tuning/nn_tuning.csv')

    print(nn_tuning.head(5))

trial 0, WAPE 10.51%, 458s
trial 1, WAPE 10.74%, 1838s
trial 2, WAPE 10.65%, 1861s
trial 3, WAPE 12.28%, 336s
trial 4, WAPE 13.80%, 314s
trial 5, WAPE 11.29%, 110s
trial 6, WAPE 11.44%, 375s
trial 7, WAPE 11.28%, 441s
trial 8, WAPE 10.21%, 438s
trial 9, WAPE 10.66%, 558s
trial 10, WAPE 11.11%, 331s
trial 11, WAPE 14.00%, 708s
trial 12, WAPE 10.47%, 252s
trial 13, WAPE 11.35%, 412s
trial 14, WAPE 10.47%, 522s
trial 15, WAPE 11.16%, 815s
trial 16, WAPE 10.84%, 2169s
trial 17, WAPE 10.15%, 623s
trial 18, WAPE 10.55%, 151s
trial 19, WAPE 10.73%, 599s
trial 20, WAPE 10.65%, 641s
trial 21, WAPE 11.22%, 299s
trial 22, WAPE 11.24%, 276s
trial 23, WAPE 11.09%, 160s
trial 24, WAPE 10.41%, 333s
trial 25, WAPE 10.46%, 192s
    trial  epochs  learning_rate  hidden_size  n_layers   dropout  \
17     17      40       0.001247          362         2  0.050092   
8       8      33       0.000473          149         3  0.068073   
24     24      21       0.004288          448         2  0.129628   
25 

In [13]:
TUNED_PARAMS = {
    'store_embed_dim': 27,
    'embed_dim': 4,
    'hidden_size': 362,
    'n_layers': 2,
    'dropout': 0.05009187597231117,
    'learning_rate': 0.0012473811256260435,
    'weight_decay': 1.2326741610799748e-06,
    'epochs': 40,
    'batch_size': 2048,
    'seed': 42,
    'n_seeds': 3
}

In [15]:
FORECAST = True

if FORECAST:

    add_tuned_nn = partial(add_nn_forecast, params=TUNED_PARAMS)

    bt = run_backtest(df_with_features, cutoffs, 7, add_tuned_nn)

    bt = bt.rename(columns={'nn_forecast': 'nn_tuned_forecast'})

    tuned_nn_bt = bt[['Store', 'Date', 'Sales', 'cutoff', 'day_ahead', 'nn_tuned_forecast']]

    tuned_nn_bt.to_csv('../results/backtests/nn_tuned_backtest.csv', index=False)

else: 
    tuned_nn_bt = pd.read_csv('../results/backtests/nn_tuned_backtest.csv', parse_dates=['Date', 'cutoff'], low_memory=False)

In [16]:
SAVE_SCORES = True

if SAVE_SCORES:
    scores = score_forecasts(tuned_nn_bt, ['nn_tuned_forecast'])
    scores.to_csv('../results/nn_tuned_scores.csv', index=False)